# **MODELOS ML Y RED NEURONAL**

In [ ]:
# ===========================
# 📌 Credit Card Fraud – Modelos ML + Red Neuronal
# ===========================

import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from imblearn.ensemble import BalancedBaggingClassifier
from imblearn.combine import SMOTETomek

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# 1) Cargar dataset
df = pd.read_csv("creditcard.csv")

# 2) Definir X, y
X = df.drop("Class", axis=1)
y = df["Class"]

print("Original distribution:", Counter(y))

# 3) Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train distribution:", Counter(y_train))

# Función para mostrar resultados
def mostrar_resultados(y_test, pred):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.show()
    print(classification_report(y_test, pred))

# ----------------------------
# 🧠 1) Modelo Base: Logistic Regression
# ----------------------------
log_base = LogisticRegression(max_iter=1000, solver="lbfgs")
log_base.fit(X_train, y_train)
print("\n=== Logistic Regression (base) ===")
mostrar_resultados(y_test, log_base.predict(X_test))

# ----------------------------
# 🧠 2) Logistic Regression con class_weight
# ----------------------------
log_bal = LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced")
log_bal.fit(X_train, y_train)
print("\n=== Logistic Regression (balanced) ===")
mostrar_resultados(y_test, log_bal.predict(X_test))

# ----------------------------
# 🧠 3) Decision Tree
# ----------------------------
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("\n=== Decision Tree ===")
mostrar_resultados(y_test, dt.predict(X_test))

# ----------------------------
# 🧠 4) Random Forest
# ----------------------------
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
print("\n=== Random Forest ===")
mostrar_resultados(y_test, rf.predict(X_test))

# ----------------------------
# 🧠 5) Balanced Bagging (Ensemble)
# ----------------------------
bbc = BalancedBaggingClassifier(
    estimator=DecisionTreeClassifier(),
    sampling_strategy='auto',
    replacement=False,
    random_state=42,
    n_estimators=50
)
bbc.fit(X_train, y_train)
print("\n=== Balanced Bagging ===")
mostrar_resultados(y_test, bbc.predict(X_test))

# ----------------------------
# 🧠 6) SMOTETomek + Logistic
# ----------------------------
sm = SMOTETomek(sampling_strategy=0.5, random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print("\nSMOTETomek train distribution:", Counter(y_res))
log_sm = LogisticRegression(max_iter=1000, solver="lbfgs")
log_sm.fit(X_res, y_res)
print("\n=== Logistic Regression (SMOTETomek) ===")
mostrar_resultados(y_test, log_sm.predict(X_test))

# ----------------------------
# 🔥 7) Red Neuronal con Keras
# ----------------------------
# Normalización simple
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Definir modelo
nn = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

nn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Entrenar
nn.fit(X_train_scaled, y_train, epochs=20, batch_size=256, verbose=1)

# Evaluar
pred_nn = (nn.predict(X_test_scaled) > 0.5).astype(int)
print("\n=== Neural Network ===")
mostrar_resultados(y_test, pred_nn)


Original distribution: Counter({0: 284315, 1: 492})
Train distribution: Counter({0: 227451, 1: 394})
